<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Copy_of_Final_XLSTM_Gannet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Libraries, Device Configuration, and Dynamic Model Architecture

In [97]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

class NativesLSTMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(NativesLSTMLayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.W_x = nn.Linear(input_dim, 4 * hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

        # Residual projection if dimensions differ
        self.residual_proj = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()

        init_weights(self.W_x)
        init_weights(self.W_h)
        if isinstance(self.residual_proj, nn.Linear):
            init_weights(self.residual_proj)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            gates = self.W_x(x_t) + self.W_h(h)
            i_gate, f_gate, c_gate, o_gate = gates.chunk(4, dim=1)

            i_t = torch.exp(torch.clamp(i_gate, -5.0, 5.0))
            f_t = torch.exp(torch.clamp(f_gate, -5.0, 5.0))

            c_tilde = torch.tanh(c_gate)
            c = f_t * c + i_t * c_tilde
            o_t = torch.sigmoid(o_gate)
            h = o_t * torch.tanh(c)

            # Apply Residual Connection per time-step
            res = self.residual_proj(x_t)
            h = h + res

            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)

class xLSTMEncoderPredictor(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim, num_layers=2, dropout_prob=0.2):
        super(xLSTMEncoderPredictor, self).__init__()
        self.num_layers = num_layers

        # Dynamic Encoder Stack
        self.encoder_layers = nn.ModuleList()
        current_dim = input_dim
        next_dim = hidden_dim
        for i in range(num_layers):
            self.encoder_layers.append(NativesLSTMLayer(current_dim, next_dim))
            current_dim = next_dim
            if i == 0 and num_layers > 1:
                next_dim = max(8, hidden_dim // 2)

        # Bottleneck and Predictor Head
        self.bottleneck = nn.Linear(current_dim, latent_dim)
        self.encoder_dropout = nn.Dropout(dropout_prob)

        # Predictor Head connects directly to the latent space
        self.predictor_head = nn.Linear(latent_dim, 1)

        init_weights(self.bottleneck)
        init_weights(self.predictor_head)

    def forward(self, x):
        # 1. Forward through Encoder Stack
        encoded = x
        for layer in self.encoder_layers:
            encoded = layer(encoded)
            encoded = self.encoder_dropout(encoded)

        # Extract the last time step's feature map for the bottleneck
        latent = self.bottleneck(encoded[:, -1, :])

        # Predict the target value (Carbon)
        predicted_carbon = self.predictor_head(latent)

        return predicted_carbon, latent

#Cell 2: Data Preprocessing and Splitting

In [98]:
FILE_PATH = '/content/rural_carbon_dataset.csv'
df = pd.read_csv(FILE_PATH)
df_processed = df.copy()

# Feature Engineering
crop_encoder = LabelEncoder()
df_processed['Crop_Type_Encoded'] = crop_encoder.fit_transform(df_processed['Crop_Type'])
df_processed['Month_sin'] = np.sin(2 * np.pi * df_processed['Month'] / 12)
df_processed['Month_cos'] = np.cos(2 * np.pi * df_processed['Month'] / 12)
df_processed['Livestock_Total'] = df_processed['Livestock_Cows'] + df_processed['Livestock_Pigs']
df_processed['Energy_per_Area'] = df_processed['Household_Energy_kWh'] / (df_processed['Crop_Area_ha'] + 1)
df_processed['Fertilizer_per_Area'] = df_processed['Fertilizer_Usage_kg'] / (df_processed['Crop_Area_ha'] + 1)

feature_cols = [
    'Month_sin', 'Month_cos', 'Crop_Type_Encoded', 'Crop_Area_ha', 'Livestock_Total',
    'Household_Energy_kWh', 'Renewable_Energy_Fraction', 'Temperature_C',
    'Rainfall_mm', 'Energy_per_Area', 'Fertilizer_per_Area'
]

X = df_processed[feature_cols].values.astype(np.float32)
y = df_processed['Carbon_Emission_tCO2'].values.astype(np.float32).reshape(-1, 1)

# Split into Train, Validation & test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=42) # 0.1765 * 0.85 ≈ 0.15

# Normalization
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

X_train_3d = np.expand_dims(X_train_scaled, axis=1)
X_val_3d = np.expand_dims(X_val_scaled, axis=1)
X_test_3d = np.expand_dims(X_test_scaled, axis=1)

val_inputs_X = torch.tensor(X_val_3d).to(device)
val_targets_y = torch.tensor(y_val_scaled).to(device)

test_inputs_X = torch.tensor(X_test_3d).to(device)
test_targets_y = torch.tensor(y_test_scaled).to(device)
feat_dim = X_train_3d.shape[2]

print("Data preprocessing with Train/Valid/Test split completed successfully.")

Data preprocessing with Train/Valid/Test split completed successfully.


#Cell 3: Gannet Optimization Algorithm Phase (Independent Metaheuristic Stage)

In [104]:
# Gannet Optimization Algorithm for Hyperparameter Tuning
# Search Space Dimensions:
# [LR, Hidden_Size, Latent_Dim, Dropout, Weight_Decay, Batch_Size, Num_Layers]
#alpha removed bec we removed the decoder
lb = np.array([1e-4, 16,  4, 0.2, 1e-6, 16,  1]) # Lower bounds
ub = np.array([1e-2, 64, 16, 0.5, 1e-3, 128, 3]) # Upper bounds

pop_size = 10
max_iter = 5

def evaluate_fitness(position):
    # Decode position values
    lr = float(position[0])
    hidden_size = int(np.round(position[1]))
    latent_dim = int(np.round(position[2]))
    dropout = float(position[3])
    weight_decay = float(position[4])
    batch_size = int(np.round(position[5]))
    num_layers = int(np.round(position[6]))

    # Fast evaluation dataloader
    t_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
    t_loader = DataLoader(t_dataset, batch_size=batch_size, shuffle=True)

    # Instantiate dynamic model
    eval_model = xLSTMEncoderPredictor(
        input_dim=feat_dim, latent_dim=latent_dim, hidden_dim=hidden_size,
        num_layers=num_layers, dropout_prob=dropout
    ).to(device)

    criterion_p = nn.MSELoss()
    opt = optim.Adam(eval_model.parameters(), lr=lr, weight_decay=weight_decay)

    # Short optimization training loop for fitness assignment
    for epoch in range(3):
        eval_model.train()
        for bx, by in t_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            p_out, _ = eval_model(bx)
            loss = criterion_p(p_out, by)
            loss.backward()
            nn.utils.clip_grad_norm_(eval_model.parameters(), 5.0)
            opt.step()

    # Evaluation on Validation set
    eval_model.eval()
    with torch.no_grad():
        v_p, _ = eval_model(val_inputs_X)
        val_loss = criterion_p(v_p, val_targets_y).item()
    return val_loss

# Initialize Gannet Population
gannet_positions = np.random.uniform(lb, ub, (pop_size, len(lb)))
fitness_scores = np.array([evaluate_fitness(p) for p in gannet_positions])

best_idx = np.argmin(fitness_scores)
best_gannet_score = fitness_scores[best_idx]
best_gannet_position = gannet_positions[best_idx].copy()

print("Executing Gannet Optimization Strategy...")
for iteration in range(max_iter):
    for i in range(pop_size):
        # Gannet Exploration and Exploitation mathematical updating mechanics
        t = 1 - (iteration / max_iter)
        c = 0.2 * (t ** 2)
        v = np.random.randn(*lb.shape)

        if np.random.rand() < 0.5:
            # Dive exploration phase
            gannet_positions[i] = gannet_positions[i] + c * v * (gannet_positions[i] - best_gannet_position)
        else:
            # Trajectory exploitation phase
            gannet_positions[i] = best_gannet_position + c * np.random.rand() * (best_gannet_position - gannet_positions[i])

        # Bound enforcement
        gannet_positions[i] = np.clip(gannet_positions[i], lb, ub)

        # Re-evaluate
        fit = evaluate_fitness(gannet_positions[i])
        if fit < fitness_scores[i]:
            fitness_scores[i] = fit
            if fit < best_gannet_score:
                best_gannet_score = fit
                best_gannet_position = gannet_positions[i].copy()

    print(f"Gannet Iteration [{iteration+1}/{max_iter}] -> Best Discovered Fitness: {best_gannet_score:.6f}")

# Extract Optimized Global Best Parameters
best_lr = float(best_gannet_position[0])
best_hidden_size = int(np.round(best_gannet_position[1]))
best_latent_dim = int(np.round(best_gannet_position[2]))
best_dropout = float(best_gannet_position[3])
best_weight_decay = float(best_gannet_position[4])
best_batch_size = int(np.round(best_gannet_position[5]))
best_num_layers = int(np.round(best_gannet_position[6]))

print("\nGannet Search Completed. Optimal Hyperparameters Parsed.")

Executing Gannet Optimization Strategy...
Gannet Iteration [1/5] -> Best Discovered Fitness: 0.015419
Gannet Iteration [2/5] -> Best Discovered Fitness: 0.015419
Gannet Iteration [3/5] -> Best Discovered Fitness: 0.015352
Gannet Iteration [4/5] -> Best Discovered Fitness: 0.015075
Gannet Iteration [5/5] -> Best Discovered Fitness: 0.015075

Gannet Search Completed. Optimal Hyperparameters Parsed.


#Cell 4: Structured Adam Training Pipeline

In [105]:
# Construct final optimized train loader
train_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)



# Build optimal architectural network
model = xLSTMEncoderPredictor(
    input_dim=feat_dim, latent_dim=best_latent_dim, hidden_dim=best_hidden_size,
    num_layers=best_num_layers, dropout_prob=best_dropout
).to(device)

criterion_pred = nn.MSELoss()

# Primary Adam Optimizer paired with Weight Decay from Gannet Phase
optimizer = optim.Adam(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 100
best_val_loss = float('inf')
patience, patience_counter = 20, 0
checkpoint_path = 'best_xlstm_predictor_checkpoint.pth'

print(f"Beginning Deep Architecture Training Sequence using Optimal Gannet Configurations...")
print("-" * 90)

for epoch in range(epochs):
    # Training Loop
    model.train()
    train_total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        pred_carbon, _ = model(batch_x)

        loss_total = criterion_pred(pred_carbon, batch_y)

        loss_total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_total_loss += loss_total.item() * batch_x.size(0)
    train_total_loss /= len(train_loader.dataset)

    # Validation Loop
    model.eval()
    with torch.no_grad():
        val_pred, _ = model(val_inputs_X)
        val_pred_loss = criterion_pred(val_pred, val_targets_y).item()

    scheduler.step(val_pred_loss)

    if val_pred_loss < best_val_loss:
        best_val_loss = val_pred_loss
        patience_counter = 0
        safe_config_tensor = torch.tensor(best_gannet_position, dtype=torch.float32)

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'best_val_loss': float(best_val_loss),
            'config_tensor': safe_config_tensor
        }, checkpoint_path)
    else:
        patience_counter += 1
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] -> Train Loss: {train_total_loss:.6f} | Val Pred Loss: {val_pred_loss:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1} to safeguard generalization.")
        break
print("-" * 90)
print("Optimized Model Training Phase Finished.")

Beginning Deep Architecture Training Sequence using Optimal Gannet Configurations...
------------------------------------------------------------------------------------------
Epoch [001/100] -> Train Loss: 0.096559 | Val Pred Loss: 0.018767
Epoch [010/100] -> Train Loss: 0.015733 | Val Pred Loss: 0.015209
Epoch [020/100] -> Train Loss: 0.015311 | Val Pred Loss: 0.015127
Epoch [030/100] -> Train Loss: 0.015177 | Val Pred Loss: 0.015220
Epoch [040/100] -> Train Loss: 0.014838 | Val Pred Loss: 0.015099
Early stopping triggered at epoch 43 to safeguard generalization.
------------------------------------------------------------------------------------------
Optimized Model Training Phase Finished.


#Cell 5: Full Performance Evaluation Metrics

In [106]:
def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

if os.path.exists(checkpoint_path):
    # Pure native PyTorch object loading - guaranteed to pass weights_only checking
    checkpoint = torch.load(checkpoint_path, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Restored optimized model state from training epoch {checkpoint['epoch']}")

model.eval()
with torch.no_grad():
    final_pred, final_latent = model(test_inputs_X)

y_true_np = test_targets_y.cpu().numpy().flatten()
y_pred_np = final_pred.cpu().numpy().flatten()

pred_r2 = r2_score(y_true_np, y_pred_np)
pred_rmse = np.sqrt(mean_squared_error(y_true_np, y_pred_np))
pred_mae = mean_absolute_error(y_true_np, y_pred_np)
pred_mape = calculate_mape(y_true_np, y_pred_np)

print("\n================== Final Gannet-Adam Hyper-Optimized TEST Metrics ==================")
print(f"Selected Architecture Layout: {best_num_layers} xLSTM Layers (Encoder-Only)")
print(f"Selected Execution Controls : LR: {best_lr:.5f} | Batch Size: {best_batch_size} | Dropout: {best_dropout:.2f}")
print("-" * 84)
print(f"Normalized Prediction Task Test R² (Carbon Y) = {pred_r2:.6f}")
print(f"Normalized Prediction Task Test RMSE          = {pred_rmse:.6f}")
print(f"Normalized Prediction Task Test MAE           = {pred_mae:.6f}")
print(f"Normalized Prediction Task Test MAPE          = {pred_mape:.2f}%")
print(f"Latent Structural Tensor Dimensionality            = {final_latent.shape}")
print("====================================================================================")

Restored optimized model state from training epoch 23

================== Final Gannet-Adam Hyper-Optimized TEST Metrics ==================
Selected Architecture Layout: 1 xLSTM Layers (Encoder-Only)
Selected Execution Controls : LR: 0.00371 | Batch Size: 83 | Dropout: 0.31
------------------------------------------------------------------------------------
Normalized Prediction Task Test R² (Carbon Y) = 0.440801
Normalized Prediction Task Test RMSE          = 0.117615
Normalized Prediction Task Test MAE           = 0.093617
Normalized Prediction Task Test MAPE          = 21.60%
Latent Structural Tensor Dimensionality            = torch.Size([450, 11])
